In [1]:
import pandas as pd

data = pd.read_csv("US_Accidents_March23_sampled_500k.csv")

data.head(3)

,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),...,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
0,A-2047758,Source2,2,2019-06-12 10:10:56,2019-06-12 10:55:58,30.641211,-91.153481,NaN,NaN,0.000,...,False,False,False,False,True,False,Day,Day,Day,Day
1,A-4694324,Source1,2,2022-12-03 23:37:14.000000000,2022-12-04 01:56:53.000000000,38.990562,-77.399070,38.990037,-77.398282,0.056,...,False,False,False,False,False,False,Night,Night,Night,Night
2,A-5006183,Source1,2,2022-08-20 13:13:00.000000000,2022-08-20 15:22:45.000000000,34.661189,-120.492822,34.661189,-120.492442,0.022,...,False,False,False,False,True,False,Day,Day,Day,Day


In [2]:
data.columns

Index(['ID', 'Source', 'Severity', 'Start_Time', 'End_Time', 'Start_Lat',
       'Start_Lng', 'End_Lat', 'End_Lng', 'Distance(mi)', 'Description',
       'Street', 'City', 'County', 'State', 'Zipcode', 'Country', 'Timezone',
       'Airport_Code', 'Weather_Timestamp', 'Temperature(F)', 'Wind_Chill(F)',
       'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Direction',
       'Wind_Speed(mph)', 'Precipitation(in)', 'Weather_Condition', 'Amenity',
       'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit', 'Railway',
       'Roundabout', 'Station', 'Stop', 'Traffic_Calming', 'Traffic_Signal',
       'Turning_Loop', 'Sunrise_Sunset', 'Civil_Twilight', 'Nautical_Twilight',
       'Astronomical_Twilight'],
      dtype='object')

` data has too many unnecessary columns. `

In [3]:
data_to_keep = data[['ID', 'Severity', 'State', 'City', 'Zipcode', 'Start_Time', 'Weather_Condition', 'Temperature(F)', 'Humidity(%)','Visibility(mi)']].copy()

In [4]:
data_to_keep.head()

,ID,Severity,State,City,Zipcode,Start_Time,Weather_Condition,Temperature(F),Humidity(%),Visibility(mi)
0,A-2047758,2,LA,Zachary,70791-4610,2019-06-12 10:10:56,Fair,77.0,62.0,10.0
1,A-4694324,2,VA,Sterling,20164-2813,2022-12-03 23:37:14.000000000,Fair,45.0,48.0,10.0
2,A-5006183,2,CA,Lompoc,93436,2022-08-20 13:13:00.000000000,Fair,68.0,73.0,10.0
3,A-4237356,2,MN,Austin,55912,2022-02-21 17:43:04,Wintry Mix,27.0,86.0,10.0
4,A-6690583,2,CA,Bakersfield,93305-2649,2020-12-04 01:46:00,Fair,42.0,34.0,10.0


`Data cleaning`

In [5]:
data_to_keep.isnull().sum()

ID                       0
Severity                 0
State                    0
City                    19
Zipcode                116
Start_Time               0
Weather_Condition    11101
Temperature(F)       10466
Humidity(%)          11130
Visibility(mi)       11291
dtype: int64

` Because the point of this project is to see patterns in accidents, I'm going to drop all nulls from Weather_Condition `

In [6]:
data_to_keep.dropna(subset = ['Weather_Condition'], inplace= True)

In [7]:
data_to_keep.isnull().sum()

ID                      0
Severity                0
State                   0
City                   17
Zipcode                 0
Start_Time              0
Weather_Condition       0
Temperature(F)       1741
Humidity(%)          2398
Visibility(mi)       1309
dtype: int64

In [8]:
data_to_keep.duplicated().sum()

np.int64(0)

In [9]:
data_to_keep.columns

Index(['ID', 'Severity', 'State', 'City', 'Zipcode', 'Start_Time',
       'Weather_Condition', 'Temperature(F)', 'Humidity(%)', 'Visibility(mi)'],
      dtype='object')

` At what time of the day are accidents most common?`
- The time info can be extracted from the Start_Time column

In [10]:
data_to_keep['Start_Time'].unique()

# some dates are formatted differently, so that needs to fixed before converting the column to datetime

array(['2019-06-12 10:10:56', '2022-12-03 23:37:14.000000000',
       '2022-08-20 13:13:00.000000000', ..., '2021-12-19 16:25:00',
       '2020-05-15 17:20:56', '2022-04-02 23:23:13'],
      shape=(476376,), dtype=object)

In [11]:
# cutting off the .00000 from the time because it ruins some values when converted to datetime
data_to_keep['Start_Time']=data_to_keep['Start_Time'].str[0:19]
data_to_keep.head()

,ID,Severity,State,City,Zipcode,Start_Time,Weather_Condition,Temperature(F),Humidity(%),Visibility(mi)
0,A-2047758,2,LA,Zachary,70791-4610,2019-06-12 10:10:56,Fair,77.0,62.0,10.0
1,A-4694324,2,VA,Sterling,20164-2813,2022-12-03 23:37:14,Fair,45.0,48.0,10.0
2,A-5006183,2,CA,Lompoc,93436,2022-08-20 13:13:00,Fair,68.0,73.0,10.0
3,A-4237356,2,MN,Austin,55912,2022-02-21 17:43:04,Wintry Mix,27.0,86.0,10.0
4,A-6690583,2,CA,Bakersfield,93305-2649,2020-12-04 01:46:00,Fair,42.0,34.0,10.0


In [12]:
data_to_keep['Start_Time']=pd.to_datetime(data_to_keep['Start_Time'])

# confirming the value time turned to datetime
data_to_keep['Start_Time'].info()  

<class 'pandas.core.series.Series'>
Index: 488899 entries, 0 to 499999
Series name: Start_Time
Non-Null Count   Dtype         
--------------   -----         
488899 non-null  datetime64[ns]
dtypes: datetime64[ns](1)
memory usage: 7.5 MB


In [13]:
# extracting the hour from Start_Time
data_to_keep['Hour'] = data_to_keep['Start_Time'].dt.hour
data_to_keep.head()

,ID,Severity,State,City,Zipcode,Start_Time,Weather_Condition,Temperature(F),Humidity(%),Visibility(mi),Hour
0,A-2047758,2,LA,Zachary,70791-4610,2019-06-12 10:10:56,Fair,77.0,62.0,10.0,10
1,A-4694324,2,VA,Sterling,20164-2813,2022-12-03 23:37:14,Fair,45.0,48.0,10.0,23
2,A-5006183,2,CA,Lompoc,93436,2022-08-20 13:13:00,Fair,68.0,73.0,10.0,13
3,A-4237356,2,MN,Austin,55912,2022-02-21 17:43:04,Wintry Mix,27.0,86.0,10.0,17
4,A-6690583,2,CA,Bakersfield,93305-2649,2020-12-04 01:46:00,Fair,42.0,34.0,10.0,1


In [14]:
bins = [0, 6, 12, 18, 24]

bin_labels = ['Twilight', 'Morning', 'Afternoon', 'Night']

data_to_keep['Time_Of_Day'] = pd.cut(data_to_keep['Hour'], bins=bins, labels = bin_labels, right = False)

# checking for correct labels. Perfect.
data_to_keep.head()

,ID,Severity,State,City,Zipcode,Start_Time,Weather_Condition,Temperature(F),Humidity(%),Visibility(mi),Hour,Time_Of_Day
0,A-2047758,2,LA,Zachary,70791-4610,2019-06-12 10:10:56,Fair,77.0,62.0,10.0,10,Morning
1,A-4694324,2,VA,Sterling,20164-2813,2022-12-03 23:37:14,Fair,45.0,48.0,10.0,23,Night
2,A-5006183,2,CA,Lompoc,93436,2022-08-20 13:13:00,Fair,68.0,73.0,10.0,13,Afternoon
3,A-4237356,2,MN,Austin,55912,2022-02-21 17:43:04,Wintry Mix,27.0,86.0,10.0,17,Afternoon
4,A-6690583,2,CA,Bakersfield,93305-2649,2020-12-04 01:46:00,Fair,42.0,34.0,10.0,1,Twilight


In [15]:
data_to_keep.groupby('Time_Of_Day')['ID'].count().sort_values(ascending=False)

C:\Users\izzat\AppData\Local\Temp\ipykernel_2368\3750395463.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  data_to_keep.groupby('Time_Of_Day')['ID'].count().sort_values(ascending=False)


Time_Of_Day
Afternoon    182181
Morning      166776
Night         90990
Twilight      48952
Name: ID, dtype: int64

`What time of day are accidents most severe?`

In [16]:
data_to_keep.groupby('Time_Of_Day')['Severity'].mean().sort_values(ascending=False)

C:\Users\izzat\AppData\Local\Temp\ipykernel_2368\505075396.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  data_to_keep.groupby('Time_Of_Day')['Severity'].mean().sort_values(ascending=False)


Time_Of_Day
Night        2.235960
Twilight     2.216273
Afternoon    2.205675
Morning      2.204382
Name: Severity, dtype: float64

`In what weather conditions are accidents most common?`

In [17]:
# Only keep these values in the weather column
severe_weather = ['Heavy Snow', 'Heavy Rain', 'Clear', 'Fog', 'Hail','Rain']
filtered_data = data_to_keep.loc[data_to_keep['Weather_Condition'].isin(severe_weather)]


filtered_data.groupby('Weather_Condition')['ID'].count().sort_values(ascending=False)

Weather_Condition
Clear         52379
Fog            6445
Rain           5501
Heavy Rain     2133
Heavy Snow      312
Hail              5
Name: ID, dtype: int64

Clear weather conditions surprisingly have the most accidents althought this could most likely be because the top contributing states have more clear days than not.

`What about in Pennsylvania (my state)?`

In [18]:
state = ["PA"]
filtered_data_PA = filtered_data.loc[data_to_keep['State'].isin(state)]
filtered_data_PA.groupby('Weather_Condition')['ID'].count().sort_values(ascending=False)

Weather_Condition
Clear         1878
Fog            357
Rain           258
Heavy Rain      96
Heavy Snow      36
Name: ID, dtype: int64

Completely identical.

`What's the relationship between the season and accidents?`
- The month needs to be extracted from Start_Time

In [19]:
# extracting the month from the Start_Time column and storing it in a category
data_to_keep['Month'] = data_to_keep['Start_Time'].dt.month

season = {12: 'Dec-Feb', 1: 'Dec-Feb', 2: 'Dec-Feb',
          3: 'March-May', 4: 'March-May', 5: 'March-May', 
          6: 'June-Aug', 7: 'June-Aug', 8: 'June-Aug',
          9: 'Sep-Nov', 10: 'Sep-Nov', 11: 'Sep-Nov'}

data_to_keep['Month_Range'] = data_to_keep['Month'].map(season)


In [20]:
data_to_keep.groupby('Month_Range')['ID'].count().sort_values(ascending=False)


Month_Range
Dec-Feb      142836
Sep-Nov      131927
March-May    108143
June-Aug     105993
Name: ID, dtype: int64

The holiday seasons show the most accidents

`Top 5 states with the most accidents`

In [21]:
top_5_states = data_to_keep.groupby("State")['ID'].count().sort_values(ascending=False).head()
top_5_states

State
CA    110753
FL     56081
TX     36764
SC     24286
NY     22440
Name: ID, dtype: int64

`Where in PA do the worst accidents happen most often?`

In [22]:
# filtering for only rows that contain PA
state = ['PA']
data_PA = data_to_keep.loc[data_to_keep['State'].isin(state)]

# filtering for only the worst case accidents
severity = [4]
data_PA = data_PA[data_PA['Severity'].isin(severity)]

data_PA.groupby('City')['ID'].count().sort_values(ascending=False).head()

City
Pittsburgh      63
Philadelphia    46
Harrisburg      21
York            19
Washington      13
Name: ID, dtype: int64

The worst case accidents happen most frequently in the densely populated cities of PA.

In [23]:
data_to_keep.to_csv('accidens_cleaned_data.csv', index= False)